# 4a_QRF_MODEL — QRF Training, Evaluation & Target-Grid Prediction

Merged notebook replacing the former split `4a_QRF_MODEL` + `5a_QRF_TARGETS`.
All variables remain in-memory — no artefact round-trip is required for target
prediction. Model files are still saved for later re-use.

**Pipeline**

| Section | Purpose |
|---|---|
| 1 | Imports & constants |
| 2 | Load IHFC reference observations |
| 3 | Load sweep parameters (`qrf_params.json`) |
| 4 | Train/test split + `StandardScaler` |
| 5 | Fit QRF (reload from pickle if present) |
| 6 | Test-set predictions + conformal calibration |
| 7 | Correction spline helpers |
| 7a | Fit & apply PCHIP correction spline |
| 8 | Metrics |
| 9 | Diagnostic figures |
| 10 | Save artefact bundle (`qrf_artefacts.pkl`) |
| 11 | Apply QRF to Antarctic & Greenland target grids |
| 12 | Summary statistics of written fields |

**Files written**
- `output/models/qrf_model.pkl` — fitted QRF estimator
- `output/models/qrf_artefacts.pkl` — full bundle (for re-use elsewhere)
- `output/models/qrf_metrics.csv`
- `fig/models/qrf_correction_spline.png`, `qrf_scatter_residuals.png`, `qrf_pi_width.png`
- `output/targets/Aq1_5_QRF_v{MODEL_VERSION}.nc` — Antarctica
- `output/targets/Kq1_5_QRF_v{MODEL_VERSION}.nc` — Greenland


## 1. Imports & constants

In [ ]:
import sys, json, pickle, time, warnings, datetime
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import PchipInterpolator
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from config import (
    obs_model, q_clip_max, random_state,
    model_dir, targets_dir, param_paths,
    netcdf_compression_level, NETCDF_AUTHOR, NETCDF_CONVENTIONS,
    PRED_QUANTILES, TARGET_GRIDS, MODEL_VERSION,
)

# ── paths ──────────────────────────────────────────────────────────────────────
local_data = Path('data')
param_dir  = Path('output/sweeps')
fig_dir    = Path('fig/models')
model_dir.mkdir(parents=True, exist_ok=True)
targets_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ── constants ──────────────────────────────────────────────────────────────────
TEST_FRAC       = 0.20
RANDOM_SEED     = 42
TARGET_COL      = 'q'
Q_CLIP_MIN      = 0.0
QUANTILES       = PRED_QUANTILES                          # [0.05, 0.25, 0.50, 0.75, 0.95]
Q50_IDX         = QUANTILES.index(0.50)
QUANTILES_DENSE = np.linspace(0.02, 0.98, 49).tolist()
HIST_BIN_W      = 0.010
CONFORMAL_ALPHA = 0.10
SPLINE_PCTLS    = 200
FIG_DPI         = 150

print(f'obs_model features : {len(obs_model)}')
print(f'q_clip_max         : {q_clip_max} W/m²')
print(f'QUANTILES          : {QUANTILES}')
print(f'MODEL_VERSION      : {MODEL_VERSION}')
print(f'targets_dir        : {targets_dir}')


## 2. Load reference data

In [ ]:
df = pd.read_parquet(local_data / 'IHFC_obs.parquet')
print(f'Raw rows: {len(df)}')

df = df[(df[TARGET_COL] >= Q_CLIP_MIN) & (df[TARGET_COL] <= q_clip_max)].copy()
df = df.dropna(subset=obs_model)
print(f'After clip & dropna : {len(df)} rows')

if 'weight' in df.columns:
    w_all = df['weight'].values.astype(np.float32)
    w_all = w_all / w_all.mean()
    print(f'Weights  min={w_all.min():.4f}  max={w_all.max():.4f}  mean={w_all.mean():.4f}')
else:
    w_all = np.ones(len(df), dtype=np.float32)
    print('WARNING: no weight column — using uniform weights')


## 3. Load sweep parameters

In [ ]:
from quantile_forest import RandomForestQuantileRegressor

with open(param_dir / 'qrf_params.json') as fp:
    PARAMS = json.load(fp)

# obs_sel: features from sweep JSON, fall back to obs_model
obs_sel = PARAMS.get('features', obs_model)
print(f'Loaded qrf_params.json')
print(f'  obs_sel ({len(obs_sel)} features): {obs_sel}')
print(f'  n_estimators     : {PARAMS["n_estimators"]}')
print(f'  max_depth        : {PARAMS["max_depth"]}')
print(f'  min_samples_leaf : {PARAMS["min_samples_leaf"]}')
print(f'  max_features     : {PARAMS["max_features"]}')


## 4. Train/test split & StandardScaler

In [ ]:
X_all = df[obs_sel].values.astype(np.float32)
y_all = df[TARGET_COL].values.astype(np.float32)

assert np.isfinite(X_all).all(), 'NaNs in X_all — check obs_sel'
assert np.isfinite(y_all).all(), 'NaNs in y_all'

X_tr_raw, X_te_raw, y_tr, y_te, w_tr, w_te = train_test_split(
    X_all, y_all, w_all, test_size=TEST_FRAC, random_state=RANDOM_SEED)

scaler = StandardScaler().fit(X_tr_raw)
X_tr   = scaler.transform(X_tr_raw).astype(np.float32)
X_te   = scaler.transform(X_te_raw).astype(np.float32)

# Held-out calibration slice (last 10 % of train — no additional leakage)
cal_frac = float(PARAMS.get('cal_frac', 0.10))
n_cal    = max(int(len(X_tr) * cal_frac), 100)
X_cal, y_cal, w_cal = X_tr[-n_cal:], y_tr[-n_cal:], w_tr[-n_cal:]

print(f'Train  : {len(y_tr):,}    Test : {len(y_te):,}    Cal slice : {n_cal:,}')
print(f'Features used : {len(obs_sel)}')


## 5. Fit QRF

Loads from `qrf_model.pkl` if it already exists. Safe to re-run.


In [ ]:
qrf_path = model_dir / 'qrf_model.pkl'
t0 = time.time()

if qrf_path.exists():
    print('QRF: loading saved model …')
    with open(qrf_path, 'rb') as fp:
        qrf = pickle.load(fp)
else:
    print('QRF: training …')
    qrf = RandomForestQuantileRegressor(
        n_estimators     = int(PARAMS['n_estimators']),
        max_depth        = PARAMS['max_depth'] if str(PARAMS['max_depth']) != 'None' else None,
        min_samples_leaf = int(PARAMS['min_samples_leaf']),
        max_features     = PARAMS['max_features'],
        n_jobs           = -1,
        random_state     = RANDOM_SEED,
    )
    nan_mask = ~np.isfinite(X_tr).all(axis=1)
    if nan_mask.any():
        print(f'  WARNING: dropping {nan_mask.sum()} NaN rows before fit')
    Xf, yf, wf = X_tr[~nan_mask], y_tr[~nan_mask], w_tr[~nan_mask]
    qrf.fit(Xf, yf, sample_weight=wf)
    with open(qrf_path, 'wb') as fp:
        pickle.dump(qrf, fp)
    print(f'  saved → {qrf_path}')

print(f'Wall time: {(time.time()-t0)/60:.1f} min')


## 6. Test-set predictions + conformal calibration

Conformal scores computed on the held-out calibration slice only.


In [ ]:
print('Predicting on test set …')
qrf_preds = qrf.predict(X_te, quantiles=QUANTILES)           # (n, 5)
qrf_dense = qrf.predict(X_te, quantiles=QUANTILES_DENSE)     # (n, 49)
qrf_q50   = qrf_preds[:, QUANTILES.index(0.50)]
qrf_q05   = qrf_preds[:, QUANTILES.index(0.05)]
qrf_q95   = qrf_preds[:, QUANTILES.index(0.95)]
qrf_mean  = qrf_dense.mean(axis=1)
qrf_std   = qrf_dense.std(axis=1)

# conformal calibration
cal_out    = qrf.predict(X_cal, quantiles=[0.05, 0.95])
cal_q05_qrf, cal_q95_qrf = cal_out[:, 0], cal_out[:, 1]
scores_qrf = np.maximum(cal_q05_qrf - y_cal, y_cal - cal_q95_qrf)
qhat_qrf   = np.quantile(scores_qrf,
                         (1 - CONFORMAL_ALPHA) * (1 + 1 / len(scores_qrf)))
qrf_conf_lo = np.clip(qrf_q05 - qhat_qrf, Q_CLIP_MIN, None)
qrf_conf_hi = np.clip(qrf_q95 + qhat_qrf, None, q_clip_max)
print(f'Conformal qhat = {qhat_qrf*1e3:.2f} mW/m²')


## 7. Correction spline helpers

In [ ]:
def weighted_percentile(vals, weights, pcts):
    """Weighted quantile (ignores NaN/inf)."""
    mask = np.isfinite(vals) & np.isfinite(weights)
    vs, ws = vals[mask], weights[mask]
    idx  = np.argsort(vs)
    cumw = np.cumsum(ws[idx]) / ws.sum()
    return np.interp(pcts / 100.0, cumw, vs[idx])

def build_correction_spline(y_pred_cal, y_true_cal, w_cal, n_pctls=SPLINE_PCTLS):
    """
    PCHIP spline mapping model predictions → empirical reference distribution.
    Fitted on calibration slice only (no test leakage).
    Addresses regression-to-the-mean / centre-of-distribution bias.
    """
    pctls      = np.linspace(1, 99, n_pctls)
    pred_p     = weighted_percentile(y_pred_cal, w_cal, pctls)
    true_p     = weighted_percentile(y_true_cal, w_cal, pctls)
    _, keep    = np.unique(pred_p, return_index=True)
    spline     = PchipInterpolator(pred_p[keep], true_p[keep], extrapolate=True)
    return spline, pred_p, true_p

def apply_spline(vals, spline):
    out    = np.full_like(vals, np.nan, dtype=np.float32)
    ok     = np.isfinite(vals)
    out[ok] = np.clip(spline(vals[ok]).astype(np.float32), Q_CLIP_MIN, q_clip_max)
    return out

print('Spline helpers defined.')


## 7a. Fit & apply correction spline

PCHIP monotone interpolation from predicted percentiles → observed percentiles.


In [ ]:
print('Fitting correction spline on calibration set …')
cal_q50_raw = qrf.predict(X_cal, quantiles=[0.50])
if cal_q50_raw.ndim == 2: cal_q50_raw = cal_q50_raw[:, 0]

qrf_spline, qrf_pred_p, qrf_true_p = build_correction_spline(
    cal_q50_raw, y_cal, w_cal)

# apply to test set
qrf_q50_corr   = apply_spline(qrf_q50, qrf_spline)
qrf_q05_corr   = apply_spline(qrf_q05, qrf_spline)
qrf_q95_corr   = apply_spline(qrf_q95, qrf_spline)
qrf_mean_corr  = apply_spline(qrf_mean, qrf_spline)

# spline plot
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(qrf_pred_p*1e3, qrf_true_p*1e3, lw=1.5, color='#2196F3', label='Spline')
ax.plot([0, q_clip_max*1e3], [0, q_clip_max*1e3], 'k--', lw=0.8, label='1:1')
ax.set_xlabel('Predicted Q [mW/m²]'); ax.set_ylabel('Corrected Q [mW/m²]')
ax.set_title('QRF centre-bias correction spline (cal set)')
ax.legend(); fig.tight_layout()
fig.savefig(fig_dir / 'qrf_correction_spline.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show(); print(f'Saved {fig_dir}/qrf_correction_spline.png')


## 8. Metrics

In [ ]:
def empirical_entropy(vals, bin_width=HIST_BIN_W):
    bins   = np.arange(Q_CLIP_MIN, q_clip_max + bin_width, bin_width)
    counts, _ = np.histogram(vals[np.isfinite(vals)], bins=bins)
    probs  = counts / counts.sum()
    return float(scipy_entropy(probs[probs > 0]))

def mean_bias(y_true, y_pred):
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    return float(np.mean(y_pred[valid] - y_true[valid]))

def eval_metrics(y_true, y_pred, y_lo=None, y_hi=None, label=''):
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    r2    = r2_score(y_true[valid], y_pred[valid])
    rmse  = float(np.sqrt(mean_squared_error(y_true[valid], y_pred[valid])))
    mae   = float(mean_absolute_error(y_true[valid], y_pred[valid]))
    bias  = mean_bias(y_true, y_pred)
    picp  = np.nan
    piw   = np.nan
    if y_lo is not None and y_hi is not None:
        cov  = np.isfinite(y_lo) & np.isfinite(y_hi)
        picp = float(np.mean((y_true[cov] >= y_lo[cov]) & (y_true[cov] <= y_hi[cov])))
        piw  = float(np.mean(y_hi[cov] - y_lo[cov]))
    H = empirical_entropy(y_pred)
    print(f'{label:30s}  R²={r2:.4f}  RMSE={rmse*1e3:.2f} mW/m²  MAE={mae*1e3:.2f}  '
          f'Bias={bias*1e3:.2f}  PICP={picp:.3f}  PI_w={piw*1e3:.1f}  H={H:.3f}')
    return dict(label=label, r2=r2, rmse_mW=rmse*1e3, mae_mW=mae*1e3,
                bias_mW=bias*1e3, picp=picp, pi_width_mW=piw*1e3,
                shannon_H=H, nan_frac=float((~valid).mean()))

print('Metrics helpers defined.')


In [ ]:
H_obs  = empirical_entropy(y_te)
m_raw  = eval_metrics(y_te, qrf_q50,       y_lo=qrf_q05, y_hi=qrf_q95,
                       label='QRF Q50 raw')
m_corr = eval_metrics(y_te, qrf_q50_corr,  y_lo=qrf_q05_corr, y_hi=qrf_q95_corr,
                       label='QRF Q50 corrected')
m_conf = eval_metrics(y_te, qrf_q50_corr,  y_lo=qrf_conf_lo, y_hi=qrf_conf_hi,
                       label='QRF Q50 corr+conformal')
print(f'Observed entropy : {H_obs:.3f} nat')

metrics_df = pd.DataFrame([m_raw, m_corr, m_conf]).round(4)
metrics_df.to_csv(model_dir / 'qrf_metrics.csv', index=False)
print(f'Saved {model_dir}/qrf_metrics.csv')


## 9. Diagnostic figures

In [ ]:
def scatter_residuals_fig(rows, suptitle, savepath):
    """
    rows: list of (y_true, y_raw, y_corr, label, color)
    3-column figure: uncorrected | corrected | residual overlay
    """
    lim  = (Q_CLIP_MIN * 1e3, q_clip_max * 1e3)
    bins = np.linspace(lim[0], lim[1], 80)
    fig, axes = plt.subplots(len(rows), 3, figsize=(18, 5 * len(rows)))
    if len(rows) == 1: axes = axes[None, :]

    for i, (yt, yraw, ycorr, label, color) in enumerate(rows):
        yt_mW, yr_mW, yc_mW = yt*1e3, yraw*1e3, ycorr*1e3
        for j, (yp, cmap, alpha, lbl) in enumerate([
                (yr_mW, 'Oranges', 0.8, 'uncorr'),
                (yc_mW, 'Blues',   0.8, 'corr')]):
            ax = axes[i, j]
            h, xe, ye = np.histogram2d(yt_mW, yp, bins=bins)
            h = np.ma.masked_where(h == 0, h)
            ax.pcolormesh(xe, ye, h.T, cmap=cmap,
                          norm=plt.matplotlib.colors.LogNorm(),
                          rasterized=True)
            ax.plot(lim, lim, 'k--', lw=0.9, label='1:1')
            r2 = r2_score(yt_mW, yp)
            ax.set_title(f'{label} — {lbl}  R²={r2:.3f}', fontsize=9)
            ax.set_xlim(lim); ax.set_ylim(lim)
            ax.set_xlabel('Observed [mW/m²]'); ax.set_ylabel('Predicted [mW/m²]')
            ax.legend(fontsize=7)

        ax = axes[i, 2]
        rr = yr_mW - yt_mW
        rc = yc_mW - yt_mW
        ax.hist(rr, bins=60, color='#FF9800', alpha=0.5, edgecolor='none',
                label=f'uncorr  bias={np.mean(rr):.1f} σ={np.std(rr):.1f}')
        ax.hist(rc, bins=60, color=color, alpha=0.65, edgecolor='none',
                label=f'corr    bias={np.mean(rc):.1f} σ={np.std(rc):.1f}')
        ax.axvline(0, color='k', lw=0.9)
        ax.set_xlabel('Residual [mW/m²]'); ax.set_ylabel('Count')
        ax.set_xlim(-200, 200); ax.set_title(f'{label} — residuals', fontsize=9)
        ax.legend(fontsize=7)

    fig.suptitle(suptitle, fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig(savepath, dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print(f'Saved {savepath}')

print('Figure helper defined.')


In [ ]:
scatter_residuals_fig(
    rows=[(y_te, qrf_q50, qrf_q50_corr, 'QRF Q50', '#2196F3')],
    suptitle='QRF — held-out test set (20%)',
    savepath=fig_dir / 'qrf_scatter_residuals.png',
)

# interval width vs observed
fig, ax = plt.subplots(figsize=(7, 4))
pi_width = (qrf_q95_corr - qrf_q05_corr) * 1e3
ax.scatter(y_te * 1e3, pi_width, s=1, alpha=0.2, c='#2196F3', rasterized=True)
ax.set_xlabel('Observed Q [mW/m²]'); ax.set_ylabel('PI90 width [mW/m²]')
ax.set_title(f'QRF corrected PI90 width  mean={pi_width.mean():.1f} mW/m²')
fig.tight_layout()
fig.savefig(fig_dir / 'qrf_pi_width.png', dpi=FIG_DPI, bbox_inches='tight')
plt.show(); print(f'Saved {fig_dir}/qrf_pi_width.png')


## 10. Save artefact bundle

All objects needed for external re-use are bundled into `qrf_artefacts.pkl`.
Downstream notebooks no longer need this — target prediction runs in §11 of
this same notebook using the in-memory variables directly — but the bundle is
kept for long-term reproducibility and ad-hoc re-use.


In [ ]:
artefacts = dict(
    model           = 'QRF',
    obs_sel         = obs_sel,
    scaler          = scaler,
    qrf             = qrf,
    qrf_spline      = qrf_spline,
    qhat_qrf        = float(qhat_qrf),
    QUANTILES       = QUANTILES,
    Q_CLIP_MIN      = Q_CLIP_MIN,
    q_clip_max      = float(q_clip_max),
    CONFORMAL_ALPHA = CONFORMAL_ALPHA,
    PARAMS          = PARAMS,
)
bundle_path = model_dir / 'qrf_artefacts.pkl'
with open(bundle_path, 'wb') as fp:
    pickle.dump(artefacts, fp)
print(f'Bundle saved → {bundle_path}')
print('Keys:', list(artefacts.keys()))


## 11. Apply QRF to target grids

Uses the in-memory `qrf`, `scaler`, `qrf_spline`, `qhat_qrf`, `obs_sel`,
`PARAMS` variables from the sections above — no pickle round-trip.

For each grid in `TARGET_GRIDS`:
1. Load parquet, verify features, check regularity
2. Scale features with train-fold scaler
3. Single `qrf.predict(X, quantiles=...)` call
4. Apply Q50 spline correction, propagate offset to other quantiles
5. Apply conformal bounds using `qhat_qrf`
6. Compute IQR50, IQR90, σ
7. Reshape 1-D → 2-D (y, x)
8. Write compressed NetCDF with full metadata


In [ ]:
# ── Grid helpers (keep local to §11 so this section can be re-run) ───────────
def load_grid(grid_cfg, obs_sel):
    label, xcol, ycol = grid_cfg['label'], grid_cfg['x_col'], grid_cfg['y_col']
    df_g = pd.read_parquet(grid_cfg['parquet'])
    print(f'\n[{label.upper()}] {len(df_g):,} rows')
    missing = [f for f in obs_sel if f not in df_g.columns]
    if missing:
        warnings.warn(f'Missing features {missing} — filling NaN')
        for f in missing: df_g[f] = np.nan
    nan_cols = {f: int(df_g[f].isna().sum()) for f in obs_sel if df_g[f].isna().any()}
    if nan_cols: print(f'  NaN counts: {nan_cols}')
    else: print(f'  All {len(obs_sel)} features OK, no NaNs')
    x_vals = np.sort(df_g[xcol].unique()).astype(np.float64)
    y_vals = np.sort(df_g[ycol].unique()).astype(np.float64)
    nx, ny = len(x_vals), len(y_vals)
    is_reg = (len(df_g) == nx * ny)
    if is_reg:
        dx, dy = np.diff(x_vals), np.diff(y_vals)
        print(f'  Regular {ny}x{nx}  dx={dx[0]:.0f}m dy={dy[0]:.0f}m')
    else:
        print(f'  Irregular {len(df_g):,} pts (expected {ny*nx:,}) — griddata fallback')
    return df_g, x_vals, y_vals, ny, nx, is_reg

def build_X_grid(df_g, obs_sel, scaler):
    X_raw  = df_g[obs_sel].values.astype(np.float32)
    finite = np.isfinite(X_raw).all(axis=1)
    X_sc   = np.full_like(X_raw, np.nan)
    if finite.any():
        X_sc[finite] = scaler.transform(X_raw[finite]).astype(np.float32)
    print(f'  finite points: {finite.sum():,} / {len(df_g):,}')
    return X_sc, finite

def to_2d(vals_1d, df_g, x_vals, y_vals, xcol, ycol, is_reg):
    ny, nx = len(y_vals), len(x_vals)
    if is_reg:
        tmp = df_g[[xcol, ycol]].copy()
        tmp['_v'] = vals_1d
        return tmp.pivot(index=ycol, columns=xcol, values='_v').values.astype(np.float32)
    from scipy.interpolate import griddata
    gx, gy = np.meshgrid(x_vals, y_vals)
    return griddata(df_g[[xcol, ycol]].values, vals_1d, (gx, gy),
                    method='nearest').astype(np.float32)

def apply_spline_grid(spline, vals):
    out = np.full(vals.shape, np.nan, dtype=np.float32)
    ok  = np.isfinite(vals)
    out[ok] = np.clip(spline(vals[ok]).astype(np.float32), Q_CLIP_MIN, q_clip_max)
    return out

def shannon_H_norm(values_2d, bin_width=HIST_BIN_W):
    v = values_2d.ravel()
    v = v[np.isfinite(v)]
    if len(v) == 0: return np.nan
    bins = np.arange(Q_CLIP_MIN, q_clip_max + bin_width, bin_width)
    counts, _ = np.histogram(v, bins=bins)
    p = counts / counts.sum()
    p = p[p > 0]
    H = float(scipy_entropy(p))
    H_max = float(np.log(len(bins) - 1))
    return round(H / H_max, 6) if H_max > 0 else 0.0

def make_ds(arrays, x_vals, y_vals, grid_cfg, params, obs_sel, model_tag,
            qhat, conformal_alpha, extra=None):
    coords = {'y': y_vals, 'x': x_vals}
    dvars = {name: xr.DataArray(arr, dims=['y','x'], coords=coords,
                                attrs={'units':'W m-2',
                                       'long_name':name.replace('_',' ')})
             for name, arr in arrays.items()}
    ds = xr.Dataset(dvars, coords=coords)
    attrs = dict(
        title           = f'{model_tag} heat-flow prediction',
        model           = model_tag,
        version         = MODEL_VERSION,
        institution     = NETCDF_AUTHOR,
        Conventions     = NETCDF_CONVENTIONS,
        crs             = grid_cfg['crs'],
        epsg            = str(grid_cfg['epsg']),
        region          = grid_cfg['label'].upper(),
        created         = datetime.datetime.utcnow().isoformat() + 'Z',
        obs_sel         = json.dumps(obs_sel),
        quantiles       = json.dumps(QUANTILES),
        q_clip_min      = float(Q_CLIP_MIN),
        q_clip_max      = float(q_clip_max),
        qhat_qrf_wm2    = float(qhat),
        conformal_alpha = float(conformal_alpha),
        params_json     = json.dumps(params, default=str),
    )
    if extra: attrs.update(extra)
    ds.attrs = attrs
    return ds

def save_nc(ds, path):
    enc = {v: {'zlib': True, 'complevel': netcdf_compression_level,
               'dtype': 'float32'} for v in ds.data_vars}
    ds.to_netcdf(path, encoding=enc)
    print(f'  Saved {path}  ({path.stat().st_size/1e6:.1f} MB)')

print('Grid helpers ready.')


In [ ]:
results_qrf = {}

for grid_cfg in TARGET_GRIDS:
    label    = grid_cfg['label']
    xcol     = grid_cfg['x_col']
    ycol     = grid_cfg['y_col']
    prefix   = 'Aq1_5' if label == 'ant' else 'Kq1_5'
    out_path = targets_dir / f'{prefix}_QRF_v{MODEL_VERSION}.nc'

    df_g, x_vals, y_vals, ny, nx, is_reg = load_grid(grid_cfg, obs_sel)
    X_sc, finite = build_X_grid(df_g, obs_sel, scaler)

    # ── QRF prediction (single pass, all quantiles) ──────────────────────
    print(f'  Predicting {len(df_g):,} points × {len(QUANTILES)} quantiles ...')
    t0 = time.time()
    q_preds = np.full((len(df_g), len(QUANTILES)), np.nan, dtype=np.float32)
    if finite.any():
        q_preds[finite] = qrf.predict(
            X_sc[finite], quantiles=QUANTILES
        ).astype(np.float32)
    print(f'  QRF predict wall time: {(time.time()-t0)/60:.1f} min')

    # clip to physical range
    q_preds = np.clip(q_preds, Q_CLIP_MIN, q_clip_max)

    # spline correction on Q50; additive offset propagated to other quantiles
    q50_raw  = q_preds[:, Q50_IDX]
    q50_corr = apply_spline_grid(qrf_spline, q50_raw)
    offset   = q50_corr - q50_raw

    arrays = {}
    qnames = ['q05', 'q25', 'q50', 'q75', 'q95']
    for qi, qn in enumerate(qnames):
        raw_1d  = q_preds[:, qi]
        corr_1d = np.clip(raw_1d + offset, Q_CLIP_MIN, q_clip_max)
        arrays[f'qrf_{qn}_raw' ] = to_2d(raw_1d,  df_g, x_vals, y_vals, xcol, ycol, is_reg)
        arrays[f'qrf_{qn}_corr'] = to_2d(corr_1d, df_g, x_vals, y_vals, xcol, ycol, is_reg)

    # conformal bounds on Q05 / Q95
    q05_conf = np.clip(q_preds[:, 0] - qhat_qrf, Q_CLIP_MIN, None).astype(np.float32)
    q95_conf = np.clip(q_preds[:, 4] + qhat_qrf, None, q_clip_max).astype(np.float32)
    arrays['qrf_q05_conf'] = to_2d(q05_conf, df_g, x_vals, y_vals, xcol, ycol, is_reg)
    arrays['qrf_q95_conf'] = to_2d(q95_conf, df_g, x_vals, y_vals, xcol, ycol, is_reg)

    # uncertainty metrics
    iqr50_raw  = q_preds[:, 3] - q_preds[:, 1]                     # Q75-Q25
    iqr90_raw  = q_preds[:, 4] - q_preds[:, 0]                     # Q95-Q05
    sigma_raw  = iqr90_raw / (2 * 1.6449)
    iqr50_corr = np.clip(iqr50_raw, 0, q_clip_max)                  # offsets cancel
    iqr90_corr = np.clip(iqr90_raw, 0, q_clip_max)
    sigma_corr = iqr90_corr / (2 * 1.6449)
    iqr90_conf = np.clip(q95_conf - q05_conf, 0, q_clip_max)

    for name, arr1d in [
        ('qrf_iqr50_raw',  iqr50_raw),  ('qrf_iqr90_raw',  iqr90_raw),
        ('qrf_sigma_raw',  sigma_raw),
        ('qrf_iqr50_corr', iqr50_corr), ('qrf_iqr90_corr', iqr90_corr),
        ('qrf_sigma_corr', sigma_corr),
        ('qrf_iqr90_conf', iqr90_conf),
    ]:
        arrays[name] = to_2d(arr1d, df_g, x_vals, y_vals, xcol, ycol, is_reg)

    H_raw  = shannon_H_norm(arrays['qrf_q50_raw'])
    H_corr = shannon_H_norm(arrays['qrf_q50_corr'])
    print(f'  Shannon H (norm): raw={H_raw:.4f}  corr={H_corr:.4f}')

    ds = make_ds(arrays, x_vals, y_vals, grid_cfg, PARAMS, obs_sel,
                 model_tag       = f'{prefix}_QRF',
                 qhat            = qhat_qrf,
                 conformal_alpha = CONFORMAL_ALPHA,
                 extra={'shannon_H_q50_raw': H_raw,
                        'shannon_H_q50_corr': H_corr,
                        'spline_applied': 'Q50 only; other quantiles shifted by Q50 offset'})
    save_nc(ds, out_path)
    results_qrf[label] = ds
    print(f'  [{label.upper()}] done.')

print('\nAll QRF target grids complete.')


## 12. Summary statistics

In [ ]:
for label, ds in results_qrf.items():
    print(f'\n[{label.upper()}] variables:')
    for v in ds.data_vars:
        arr = ds[v].values
        fin = arr[np.isfinite(arr)]
        if len(fin) == 0:
            print(f'  {v:30s}  (all NaN)')
        else:
            print(f'  {v:30s}  mean={fin.mean()*1e3:6.1f}  '
                  f'std={fin.std()*1e3:5.1f}  '
                  f'[{fin.min()*1e3:.1f}, {fin.max()*1e3:.1f}] mW/m²')
